<!-- # <span style="color:red">UNDER CONSTRUCTION!!!!</span> -->

# Spoken Language Processing - Instituto Superior Técnico
### Laboratory Assignment 2 - Automatic Age Estimation Challenge
<!--[image](imgs/lab2_slp_banner.png)-->
<img src="imgs/lab2_slp_banner.png" alt="drawing" width="400"/>

# WEEK 2 - Using pre-trained models


During this week, students will implement two modern systems for age regression based on:
- speaker representations (utterance-based) obtained with an x-vector model (this notebook);
- speech representations (frame-based) obtained with a self-supervised learning (SSL) pre-trained model (`lab2_ssl.ipynb` notebook).

In both cases, students are encouraged to explore different feature configurations and alternative downstream models.

## Before starting

Let's import some modules and make some definitions. 

**WARNING from professors** We changed the pf_tools.py script for this second week. Be sure to update (the new one is compatible with Week 1 lab)

In [22]:
import os
import csv
import pickle
import numpy as np
import librosa
import torch

from pf_tools import CheckThisCell, SLPdata
from speechbrain.inference.classifiers import EncoderClassifier
from speechbrain.utils.data_utils import split_path
from sklearn.svm import LinearSVC, SVR
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import matplotlib.pyplot as plt


GENDER_CLASSES = ('F',  'M')
GEN2ID = {'F':0, 'M':1}
ID2GEN = dict((GEN2ID[k],k)for k in GEN2ID)

Like in the previous Notebooks, you need to mount Google drive if you are working on Google Colab. Otherwise, you should skip or delete the following code cell:


Like in week1, the audio data is expected to be in a folder with the following format:

```
lab2_data/
├── train/
│   └── wav/
│       └──wav files
│   └── info.csv
│
└── train_small/
    └── wav/
        └──wav files
    └── info.csv
...
```

You must already have this from the previous week, so you can set-up your data directory:

In [23]:
import os

CWD = os.getcwd() 
DATADIR = os.path.abspath(os.path.join(CWD, '..', 'data'))

# Create the data directory if it doesn't exist
if not os.path.isdir(DATADIR):
    os.makedirs(DATADIR)
    print(f"Created directory: {DATADIR}")

print(f'Current working directory is set to: {CWD}')   
print(f'Your LAB2 data folder is: {DATADIR}')

Current working directory is set to: /home/luispma/slp_lab2/src
Your LAB2 data folder is: /home/luispma/slp_lab2/data


## Using pre-trained speaker embeddings (x-vectors)

The goal of this part of the lab is to become familiar with and show how to use pre-trained speaker embedings (a.k.a. x-vectors) for speech classification/regressions tasks.

There exist plenty of resources and pre-trained models that can be  useful for our task. In particular, x-vectors are the current state of the art approach to obtain speech embeddings that characterize very efficiently speaker or language, among others. X-vectors are neural models typically trained for speaker identification in a supervised way, but also in some cases for other related tasks. Once trained, they can be used to obtain a single embedding vector of fixed dimension for each audio input. This vector corresponds to the activations of one of the layers after the pooling layer.

The following are examples of x-vector models available in the `speechbrain` module:

- `speechbrain/spkrec-xvect-voxceleb`: same with a different architecture: https://huggingface.co/speechbrain/spkrec-xvect-voxceleb

- `speechbrain/spkrec-ecapa-voxceleb`: trained using a large speaker corpus for speaker verification: https://huggingface.co/speechbrain/spkrec-ecapa-voxceleb



The following code cell shows how to import one of those models to obtain an embedding vector:

In [24]:
# Instantiate the model. If you don't have GPU available, run this line
xvector_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir=f"{CWD}/tmp")

# If you have GPU available, run this line instead
# xvector_model = EncoderClassifier.from_hparams(source="speechbrain/spkrec-xvect-voxceleb", savedir=f"{CWD}/tmp", run_opts={"device":"cuda"})

signal = xvector_model.load_audio(f'{DATADIR}/train/wav/00834c0e904d40eda496e55010acebc5.wav')
emb =  xvector_model.encode_batch(signal)

print(type(emb), emb.shape)

<class 'torch.Tensor'> torch.Size([1, 1, 512])


These (very informative) embedding vectors can be used to train simple models for several speech classification tasks, achieving excellent results. In particular, in this lab assignment, we will train a simple Support Vector Regression (SVR) on top of these x-vectors.


Student groups will be graded depending on their ability to explore different feature configurations and model alternatives/configurations.

### 1. Extracting x-vectors for the SLP datasets

Just like in Part1, we will code the feature transformation to process all data and obtain x-vectors. In this case, the function should receive as arguments the audio filename and an instance of `EncoderClassifier` (the x-vector model) and return the numpy array with the features. You must complete the following code using the previous example:

In [25]:
def extract_xvec(filename, emb_model):
    """
    Extract x-vector embedding from an audio file.
    Returns numpy array of shape (1, D).
    """
    # Load audio using the model's own loader (handles resampling)
    signal = emb_model.load_audio(filename)          # (T,) tensor
 
    # encode_batch expects (batch, T) — unsqueeze adds batch dim
    embedding = emb_model.encode_batch(signal.unsqueeze(0))  # (1, 1, D)
 
    # Squeeze to (1, D) and move to CPU numpy
    embedding = embedding.squeeze(1).detach().cpu().numpy()  # (1, D)
 
    # Remove the soft-link created by speechbrain
    _, fl = split_path(filename)
    if os.path.islink(fl):
        os.remove(fl)
 
    return embedding   # shape (1, D)
 
 
# Quick sanity check
emb = extract_xvec(
    f'{DATADIR}/train/wav/00834c0e904d40eda496e55010acebc5.wav',
    xvector_model
)
print(emb.shape, type(emb))   # expect (1, 512) <class 'numpy.ndarray'>


(1, 512) <class 'numpy.ndarray'>


Let's generate the x-vectors for all our data sets using the SLP class and store in disk. Like in Part 1, we can keep different transformation configurations in a dictionary for later usage.  

Let's define first our configurations (you can try to different x-vector models, the propsed ones or even other that you may find in huggingface):

In [26]:
import librosa
import soundfile as sf
import numpy as np
import torch
from transformers import AutoFeatureExtractor, WavLMModel, Wav2Vec2Processor, Wav2Vec2Model
 
TARGET_SR = 16000   # all models expect 16 kHz
 
 
# ── HuggingFace helper: mean-pool hidden states → (1, D) numpy ─
 
def extract_hf_embedding(filename, processor, model, layer=-1):
    """
    Load audio, run through a HuggingFace wav2vec2/WavLM model,
    and return mean-pooled hidden states as (1, D) numpy array.
    layer=-1 uses the last hidden state; change to an int index
    to tap an intermediate layer.
    """
    audio, sr = sf.read(filename)
    if audio.ndim > 1:          # stereo → mono
        audio = audio.mean(axis=1)
    if sr != TARGET_SR:
        audio = librosa.resample(audio, orig_sr=sr, target_sr=TARGET_SR)
 
    inputs = processor(audio, sampling_rate=TARGET_SR,
                       return_tensors="pt", padding=True)
 
    with torch.no_grad():
        outputs = model(**inputs, output_hidden_states=True)
 
    # Pick layer and mean-pool over time
    hidden = outputs.hidden_states[layer]   # (1, T, D)
    emb    = hidden.mean(dim=1)             # (1, D)
    return emb.cpu().numpy()
 
 
# ── 1 & 2: speechbrain models (same as before) ─────────────────
 
transform = {
    'spkrec-ecapa-voxceleb': {
        'audio_transform': lambda x: extract_xvec(
            x,
            emb_model=EncoderClassifier.from_hparams(
                source="speechbrain/spkrec-ecapa-voxceleb",
                savedir=f"{CWD}/tmp/spkrec-ecapa-voxceleb"
            )
        ),
        'chunk_transform': None,
        'chunk_size': 0,
        'chunk_hop': 0
    }
}
 
transform['spkrec-xvect-voxceleb'] = {
    'audio_transform': lambda x: extract_xvec(
        x,
        emb_model=EncoderClassifier.from_hparams(
            source="speechbrain/spkrec-xvect-voxceleb",
            savedir=f"{CWD}/tmp/spkrec-xvect-voxceleb"
        )
    ),
    'chunk_transform': None,
    'chunk_size': 0,
    'chunk_hop': 0
}
 
# ── 3: WavLM-base-plus speaker verification (HuggingFace) ───────
#   microsoft/wavlm-base-plus-sv  — 256-d output
#   Strong speaker-discriminative features; captures voice timbre
#   changes correlated with age.
 
_wavlm_proc  = AutoFeatureExtractor.from_pretrained(
    "microsoft/wavlm-base-plus-sv")
_wavlm_model = WavLMModel.from_pretrained(
    "microsoft/wavlm-base-plus-sv").eval()
 
transform['wavlm-base-plus-sv'] = {
    'audio_transform': lambda x: extract_hf_embedding(
        x, _wavlm_proc, _wavlm_model, layer=-1
    ),
    'chunk_transform': None,
    'chunk_size': 0,
    'chunk_hop': 0
}
 
# ── 4: wav2vec2-large fine-tuned for age+gender (HuggingFace) ───
#   audeering/wav2vec2-large-robust-24-ft-age-gender — 1024-d
#   Pre-trained on Voxceleb specifically for age & gender.
#   Using the 2nd-to-last hidden layer as a rich age-aware embedding
#   rather than the final head, so we can still train our own regressor.
 
_age_proc  = Wav2Vec2Processor.from_pretrained(
    "audeering/wav2vec2-large-robust-24-ft-age-gender")
_age_model = Wav2Vec2Model.from_pretrained(
    "audeering/wav2vec2-large-robust-24-ft-age-gender").eval()
 
transform['wav2vec2-age-gender'] = {
    'audio_transform': lambda x: extract_hf_embedding(
        x, _age_proc, _age_model, layer=-2   # penultimate layer
    ),
    'chunk_transform': None,
    'chunk_size': 0,
    'chunk_hop': 0
}
 
print("Defined transforms:", list(transform.keys()))

Loading weights: 100%|██████████| 248/248 [00:00<00:00, 21753.51it/s]
[transformers] WavLMModel LOAD REPORT from: microsoft/wavlm-base-plus-sv
Key                                | Status     |  | 
-----------------------------------+------------+--+-
feature_extractor.bias             | UNEXPECTED |  | 
tdnn.{0, 1, 2, 3, 4}.kernel.bias   | UNEXPECTED |  | 
feature_extractor.weight           | UNEXPECTED |  | 
tdnn.{0, 1, 2, 3, 4}.kernel.weight | UNEXPECTED |  | 
projector.bias                     | UNEXPECTED |  | 
objective.weight                   | UNEXPECTED |  | 
projector.weight                   | UNEXPECTED |  | 
classifier.weight                  | UNEXPECTED |  | 
layer_weights                      | UNEXPECTED |  | 
classifier.bias                    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 422/422 [00:00<00:00, 423.74it/s]
[transformers] Wav2V

Defined transforms: ['spkrec-ecapa-voxceleb', 'spkrec-xvect-voxceleb', 'wavlm-base-plus-sv', 'wav2vec2-age-gender']


And now let's do feature extraction. Be patient because this process can be a bit slow depending on the resources of your machine (train_small without GPU should take around 5min on google colab):

In [27]:
# Download and feature extract
trainset = 'train'
transform_id = 'wav2vec2-age-gender'  # choose from the keys of the transform dict

slp_partitions = {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train', 'dev', 'evl'):
    slp_partitions[partition] = SLPdata(DATADIR, partition,
                    transform_id=transform_id,
                    audio_transform=transform[transform_id]['audio_transform'],
                    chunk_transform=transform[transform_id]['chunk_transform'],
                    chunk_size=transform[transform_id]['chunk_size'],
                    chunk_hop=transform[transform_id]['chunk_hop']
                    )

0it [00:00, ?it/s]

3238it [4:53:25,  5.44s/it]
/home/luispma/slp_lab2/src/pf_tools.py:108: UserWarning: The feature directory already exists, and no new feature extraction will be performed.
  warnings.warn("The feature directory already exists, and no new feature extraction will be performed.")
/home/luispma/slp_lab2/src/pf_tools.py:108: UserWarning: The feature directory already exists, and no new feature extraction will be performed.
  warnings.warn("The feature directory already exists, and no new feature extraction will be performed.")


### 2. Training an SVR model

Our first attempt of age regression system based on x-vectors will be a simple SVR model like in the `openSMILE` baseline, but in this case we will be using x-vectors as features.

First, we will use the SLP data instances to store the x-vectors, the labels and file identifiers in numpy arrays:

In [28]:
from pf_tools import prepare_slp_data

#   Concatenate all data and labels
#   Each row corresponds to a file
#   We store the data, labels and file identifiers of each partition in dictionaries
#     with the partition name as key


gender_label_pos = 0
age_label_pos = 1

data, labels_gender, labels_age, fileids = {}, {}, {}, {}
# for partition in ('train', 'train_small', 'dev', 'evl'):
for partition in ('train', 'dev', 'evl'):
    data_and_labels = prepare_slp_data(slp_partitions[partition])
    print(f'Partition: {partition}')
    print(f'Number of samples: {data_and_labels["data"].shape[0]}')
    print(f'Number of features: {data_and_labels["data"].shape[1]}')
    print(f'Number of labels: {len(np.unique(data_and_labels["label"][:,gender_label_pos]))}')
    print(f'Number of identifiers (samples): {len(np.unique(data_and_labels["identifiers"]))}')
    print('---')
    data[partition] = data_and_labels['data']
    labels_gender[partition] = data_and_labels['label'][:,gender_label_pos]
    labels_age[partition] = data_and_labels['label'][:,age_label_pos]
    fileids[partition] = data_and_labels['identifiers']



Partition: train
Number of samples: 3238
Number of features: 1024
Number of labels: 2
Number of identifiers (samples): 3238
---
Partition: dev
Number of samples: 117
Number of features: 1024
Number of labels: 2
Number of identifiers (samples): 117
---
Partition: evl
Number of samples: 145
Number of features: 1024
Number of labels: 1
Number of identifiers (samples): 145
---


Now, we will use `sklearn` Support Vector Regression (SVR) to:
1. Train our regressor and save it for later use.
2. Predict on the dev and evl partitions and save the results

In [ ]:
from sklearn.svm import LinearSVR
from pf_tools import save_model
import time


trainset = 'train'

print(f"Training LinearSVR on {data[trainset].shape[0]} samples, {data[trainset].shape[1]} features")
print(f"Started at: {time.strftime('%H:%M:%S')}")
t0 = time.time()

model = LinearSVR(max_iter=5000, verbose=1)
model.fit(data[trainset], labels_age[trainset])

print(f"Training done in {(time.time()-t0)/60:.1f} min")

model_id = save_model(model, f'svr_{transform_id}', f'{DATADIR}/{trainset}/models/')
print(f'Model {model_id} saved in {DATADIR}/{trainset}/models/')

t0 = time.time()
print("Predicting dev...")
dev_results = model.predict(data['dev'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl'
pickle.dump({'hyp': dev_results, 'fileids': fileids['dev']}, open(filename, 'wb'))
print(f"Dev done in {(time.time()-t0):.1f}s")

t0 = time.time()
print("Predicting evl...")
evl_results = model.predict(data['evl'])
filename = f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl'
pickle.dump({'hyp': evl_results, 'fileids': fileids['evl']}, open(filename, 'wb'))
print(f"Evl done in {(time.time()-t0):.1f}s")

It should be extremely easy to experiment other models provided in the `sklearn` module, including SVMs with other kernels, Random Forests, etc.


#### 2.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

ref, hyp = labels_age['dev'], dev_results

print(f'Mean Absolute Error: {mean_absolute_error(ref, hyp):.2f}')
print(f'Mean Squared Error: {mean_squared_error(ref, hyp):.2f}')

Mean Absolute Error: 10.17
Mean Squared Error: 164.07


You should obtain a mean absolute error around 7.6 (it will depend on the xvector model chosen). 

Now, let's generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

model_id = 'svr_spkrec-xvect-voxceleb_2026-05-14_01:46:15'
model_id_short = 'svr_spkrec-xvect'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)


At this point, you can explore different x-vector model configurations for feature extraction and alternative models to the SVR.

### 3. Training a neural network model

As an alternative to the SVR, we will explore simple neural models on top of x-vector features.

We will need to define the size of the feature vector that will be used as input to the neural network:

In [1]:
feat_dim = data[trainset].shape[1]
print(f"transform_id : {transform_id}")
print(f"feat_dim     : {feat_dim}")

NameError: name 'data' is not defined

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

imputer = SimpleImputer(strategy='mean')
scaler  = StandardScaler()

X_tr_clean = scaler.fit_transform(imputer.fit_transform(X_tr))
X_dv_clean = scaler.transform(imputer.transform(X_dv))
X_ev_clean = scaler.transform(imputer.transform(data['evl']))

print(f"After cleaning:")
print(f"X_train  NaN: {np.isnan(X_tr_clean).sum()}  max={X_tr_clean.max():.3f}")
print(f"X_dev    NaN: {np.isnan(X_dv_clean).sum()}  max={X_dv_clean.max():.3f}")

After cleaning:
X_train  NaN: 0  max=5.690
X_dev    NaN: 0  max=5.452


And a simple neural model architecture (you can change this):

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
import pickle
 
class ResBlockLN(nn.Module):
    """Residual block using LayerNorm — safe for any batch size."""
    def __init__(self, in_dim, out_dim, dropout=0.3):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
            nn.GELU(),
        )
        self.proj = (nn.Linear(in_dim, out_dim, bias=False)
                     if in_dim != out_dim else nn.Identity())
 
    def forward(self, x):
        return self.block(x) + self.proj(x)
 
 
class AgeRegressorFixed(nn.Module):
    def __init__(self, input_dim, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            ResBlockLN(256, 256, dropout),
            ResBlockLN(256, 128, dropout),
            ResBlockLN(128,  64, dropout),
            nn.Linear(64, 1),
        )
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
 
    def forward(self, x):
        out = self.net(x)
        if not self.training:
            out = out.clamp(18, 80)
        return out


We will use a simple `train_nn` function included in the `pf_tools` script that will permit training the model using backpropagation. Students are encouraged to explore this function and, eventually, to modify it to experiment alternative training strategies, parameters, etc.

In [ ]:
# ================================================================
# MLP FULL GRID SEARCH (all combinations)
# ================================================================
# Requires: X_tr_clean, X_dv_clean, y_tr, y_dv, feat_dim, device
# ================================================================

import itertools
import time
import random
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import mean_absolute_error
import pickle

# ── Search space ─────────────────────────────────────────────────

search_space = {
    'architecture': [
        [256, 256, 128, 64],
        [512, 256, 128, 64],
        [512, 512, 256, 128, 64],
        [128, 128, 64],
    ],
    'dropout':      [0.2, 0.3, 0.5],
    'lr':           [1e-3, 3e-4, 1e-4],
    'weight_decay': [1e-3, 1e-4],
    'huber_delta':  [3.0, 5.0, 10.0],
    'batch_size':   [32, 64],
}

keys   = list(search_space.keys())
combos = list(itertools.product(*search_space.values()))
trials = combos          # all 432 combinations
N_TRIALS = len(trials)

# ── Search hyperparams ────────────────────────────────────────────
# Shorter per-trial budget; winner gets full 500 epochs later.
MAX_EPOCH = 100
PATIENCE  = 25

print(f"Running ALL {N_TRIALS} combinations "
      f"(max {MAX_EPOCH} ep / trial, patience {PATIENCE})\n")


# ── Model definition ─────────────────────────────────────────────

class ResBlockLN(nn.Module):
    def __init__(self, in_dim, out_dim, dropout):
        super().__init__()
        self.block = nn.Sequential(
            nn.LayerNorm(in_dim),
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
            nn.GELU(),
        )
        self.proj = (nn.Linear(in_dim, out_dim, bias=False)
                     if in_dim != out_dim else nn.Identity())

    def forward(self, x):
        return self.block(x) + self.proj(x)


def build_model(input_dim, arch, dropout):
    layers = [nn.Linear(input_dim, arch[0]), nn.GELU(), nn.Dropout(dropout)]
    for i in range(len(arch) - 1):
        layers.append(ResBlockLN(arch[i], arch[i + 1], dropout))
    layers.append(nn.Linear(arch[-1], 1))
    net = nn.Sequential(*layers)
    for m in net.modules():
        if isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, nonlinearity='relu')
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return net


# ── Precompute tensors once ───────────────────────────────────────

X_t    = torch.FloatTensor(X_tr_clean)
y_t    = torch.FloatTensor(y_tr).unsqueeze(1)
X_v    = torch.FloatTensor(X_dv_clean).to(device)
y_v_np = y_dv


# ── Search loop ──────────────────────────────────────────────────

results        = []
total_start    = time.time()
trial_times    = []

for t_idx, combo in enumerate(trials):
    trial_start = time.time()
    cfg = dict(zip(keys, combo))
    arch, dropout, lr, wd, delta, bs = (
        cfg['architecture'], cfg['dropout'], cfg['lr'],
        cfg['weight_decay'], cfg['huber_delta'], cfg['batch_size']
    )

    loader = DataLoader(TensorDataset(X_t, y_t),
                        batch_size=bs, shuffle=True, drop_last=True)

    net  = build_model(feat_dim, arch, dropout).to(device)
    opt  = optim.AdamW(net.parameters(), lr=lr, weight_decay=wd)
    sch  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCH)
    crit = nn.HuberLoss(delta=delta)

    best_mae, patience_cnt = float('inf'), 0

    for epoch in range(1, MAX_EPOCH + 1):
        net.train()
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(net(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            opt.step()
        sch.step()

        net.eval()
        with torch.no_grad():
            hyp = net(X_v).clamp(18, 80).cpu().numpy().ravel()
        mae = mean_absolute_error(y_v_np, hyp)

        if mae < best_mae:
            best_mae      = mae
            patience_cnt  = 0
        else:
            patience_cnt += 1
        if patience_cnt >= PATIENCE:
            break

    results.append((best_mae, cfg, epoch))

    # ── Progress / ETA ───────────────────────────────────────────
    trial_elapsed = time.time() - trial_start
    trial_times.append(trial_elapsed)
    avg_t     = sum(trial_times) / len(trial_times)
    remaining = avg_t * (N_TRIALS - t_idx - 1)
    eta_min   = remaining / 60

    arch_str = str(arch)
    print(f"[{t_idx + 1:>3}/{N_TRIALS}] MAE={best_mae:.2f}  "
          f"arch={arch_str:<30} drop={dropout}  lr={lr}  "
          f"wd={wd}  delta={delta}  bs={bs}  "
          f"ep={epoch}  ETA={eta_min:.0f}m")


# ── Leaderboard ──────────────────────────────────────────────────

results.sort(key=lambda x: x[0])
total_min = (time.time() - total_start) / 60

print(f"\n{'='*60}")
print(f"  GRID SEARCH COMPLETE — {total_min:.0f} min total")
print(f"{'='*60}")
print(f"\n  TOP 5 CONFIGS")
print(f"{'='*60}")
for rank, (mae, cfg, ep) in enumerate(results[:5], 1):
    print(f"\n  #{rank}  MAE = {mae:.2f} years  (stopped ep {ep})")
    for k, v in cfg.items():
        print(f"       {k:<15}: {v}")

best_mae, best_cfg, _ = results[0]
print(f"\n{'='*60}")
print(f"  Best config : {best_cfg}")
print(f"  Best dev MAE: {best_mae:.2f} years")
print(f"{'='*60}")


# ================================================================
# RETRAIN BEST CONFIG FOR 500 EPOCHS
# ================================================================

print("\nRetraining best config for 500 epochs...")

arch    = best_cfg['architecture']
dropout = best_cfg['dropout']
lr      = best_cfg['lr']
wd      = best_cfg['weight_decay']
delta   = best_cfg['huber_delta']
bs      = best_cfg['batch_size']

loader = DataLoader(TensorDataset(X_t, y_t),
                    batch_size=bs, shuffle=True, drop_last=True)

best_model   = build_model(feat_dim, arch, dropout).to(device)
opt          = optim.AdamW(best_model.parameters(), lr=lr, weight_decay=wd)
sch          = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=500)
crit         = nn.HuberLoss(delta=delta)

best_mae, best_state, patience_cnt = float('inf'), None, 0

for epoch in range(1, 501):
    best_model.train()
    for xb, yb in loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        loss = crit(best_model(xb), yb)
        loss.backward()
        nn.utils.clip_grad_norm_(best_model.parameters(), 5.0)
        opt.step()
    sch.step()

    best_model.eval()
    with torch.no_grad():
        hyp = best_model(X_v).clamp(18, 80).cpu().numpy().ravel()
    mae = mean_absolute_error(y_v_np, hyp)

    if mae < best_mae:
        best_mae     = mae
        best_state   = {k: v.cpu().clone()
                        for k, v in best_model.state_dict().items()}
        patience_cnt = 0
    else:
        patience_cnt += 1

    if epoch % 100 == 0:
        print(f"  Epoch {epoch}/500  dev_MAE={mae:.2f}  best={best_mae:.2f}")

    if patience_cnt >= 50:
        print(f"  Early stop at epoch {epoch}")
        break

best_model.load_state_dict(best_state)
best_model.eval()
print(f"\nFinal best dev MAE: {best_mae:.2f} years")


# ── Save ─────────────────────────────────────────────────────────

from pf_tools import save_model

model_id = save_model(best_model, f'nnet_tuned_{transform_id}',
                      f'{DATADIR}/{trainset}/models/')

X_ev = torch.FloatTensor(X_ev_clean).to(device)

with torch.no_grad():
    hyp_dev = best_model(X_v).clamp(18, 80).cpu().numpy().ravel()
    hyp_evl = best_model(X_ev).clamp(18, 80).cpu().numpy().ravel()

pickle.dump({'hyp': hyp_dev, 'fileids': fileids['dev']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl', 'wb'))
pickle.dump({'hyp': hyp_evl, 'fileids': fileids['evl']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl', 'wb'))

print(f"Saved as : {model_id}")
print(f"Dev MAE  : {mean_absolute_error(y_dv, hyp_dev):.2f}")

Running 30 trials out of 432 possible configs

[ 1/30] MAE=5.49  arch=[128, 128, 64]                 drop=0.5  lr=0.0003  wd=0.0001  delta=5.0  bs=64  (ep=142)
[ 2/30] MAE=5.70  arch=[512, 512, 256, 128, 64]       drop=0.2  lr=0.0003  wd=0.001  delta=3.0  bs=32  (ep=69)
[ 3/30] MAE=5.60  arch=[512, 512, 256, 128, 64]       drop=0.3  lr=0.0001  wd=0.001  delta=5.0  bs=32  (ep=65)
[ 4/30] MAE=5.10  arch=[256, 256, 128, 64]            drop=0.5  lr=0.0003  wd=0.0001  delta=5.0  bs=32  (ep=104)
[ 5/30] MAE=6.36  arch=[512, 256, 128, 64]            drop=0.2  lr=0.0001  wd=0.0001  delta=3.0  bs=64  (ep=68)
[ 6/30] MAE=7.16  arch=[128, 128, 64]                 drop=0.5  lr=0.0001  wd=0.001  delta=3.0  bs=64  (ep=82)
[ 7/30] MAE=5.59  arch=[256, 256, 128, 64]            drop=0.2  lr=0.0003  wd=0.001  delta=5.0  bs=32  (ep=60)
[ 8/30] MAE=5.19  arch=[128, 128, 64]                 drop=0.3  lr=0.001  wd=0.001  delta=3.0  bs=64  (ep=114)
[ 9/30] MAE=5.97  arch=[256, 256, 128, 64]            drop=0

#### 3.1 Analyze results on the dev set and prepare your submission file

Let's check our performance on the dev set:

In [ ]:
from pf_tools import save_model
 
model_id = save_model(model, f'nnet_fixed_{transform_id}',
                      f'{DATADIR}/{trainset}/models/')
 
with torch.no_grad():
    hyp_dev = model(X_v).cpu().numpy().ravel()
    hyp_evl = model(
        torch.FloatTensor(X_ev_clean).to(device)
    ).cpu().numpy().ravel()
 
pickle.dump({'hyp': hyp_dev, 'fileids': fileids['dev']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/dev.pkl', 'wb'))
pickle.dump({'hyp': hyp_evl, 'fileids': fileids['evl']},
            open(f'{DATADIR}/{trainset}/models/{model_id}/evl.pkl', 'wb'))
 
print(f"Model saved as: {model_id}")
print(f"Dev MAE:  {mean_absolute_error(y_dv, hyp_dev):.2f}")


Model saved to /home/luispma/slp_lab2/data/train_small/models//nnet_fixed_wav2vec2-age-gender_2026-05-19_23:41:45/model.pkl
Model saved as: nnet_fixed_wav2vec2-age-gender_2026-05-19_23:41:45
Dev MAE:  5.48


And generate the final prediction file and make a submission to the  [Kaggle competition](https://www.kaggle.com/t/8d80747e0c474688a83024aabdfe1ab0):

In [ ]:
from pf_tools import create_submission_file

students_group = '00' # <--- CHANGE THIS ACCORDINGLY

# model_id = 'nnet_spkrec-xvect-voxceleb_2026-05-14_02:05:38'
model_id_short = 'nnet_spkrec-xvect-voxceleb'

results_path = f'{DATADIR}/{trainset}/models/{model_id}/'
filename = f'{CWD}/g{students_group}_{trainset}_{model_id_short}.csv' # <--- CHANGE THIS ACCORDINGLY

create_submission_file(results_path, filename)

At this point, you can explore different x-vector model configurations for feature extraction and alternative  neural model architectures and parameters.

# Contacts and support
You can contact the professors during the classes or the office hours.

Particularly, for this second laboratory assignment, you should contact Prof. Alberto Abad: alberto.abad@tecnico.ulisboa.pt


